In [25]:
import pandas as pd

In [27]:
df = pd.read_csv(
    "resources/dpe_hauts_de_france_2025-01-01_to_2026-06-04.csv",
    low_memory=False
)

print(df.shape)
print(df.columns.tolist())
print(df.head())

(336178, 46)
['etiquette_dpe', 'numero_dpe', 'date_etablissement_dpe', 'code_region_ban', 'code_departement_ban', 'code_postal_ban', 'code_insee_ban', 'nom_commune_ban', 'zone_climatique', 'classe_altitude', 'type_batiment', 'methode_application_dpe', 'periode_construction', 'annee_construction', 'surface_habitable_immeuble', 'nombre_niveau_immeuble', 'nombre_appartement', 'numero_etage_appartement', 'hauteur_sous_plafond', 'classe_inertie_batiment', 'qualite_isolation_enveloppe', 'qualite_isolation_murs', 'qualite_isolation_menuiseries', 'qualite_isolation_plancher_bas', 'qualite_isolation_plancher_haut_comble_perdu', 'type_installation_chauffage', 'type_installation_chauffage_n1', 'configuration_installation_chauffage_n1', 'type_generateur_chauffage_principal', 'type_generateur_n1_installation_n1', 'type_energie_principale_chauffage', 'type_energie_generateur_n1_installation_n1', 'type_emetteur_installation_chauffage_n1', 'usage_generateur_n1_installation_n1', 'type_installation_ecs'

In [28]:
df.dtypes

etiquette_dpe                                       str
numero_dpe                                          str
date_etablissement_dpe                              str
code_region_ban                                   int64
code_departement_ban                              int64
code_postal_ban                                 float64
code_insee_ban                                    int64
nom_commune_ban                                     str
zone_climatique                                     str
classe_altitude                                     str
type_batiment                                       str
methode_application_dpe                             str
periode_construction                                str
annee_construction                              float64
surface_habitable_immeuble                      float64
nombre_niveau_immeuble                          float64
nombre_appartement                              float64
numero_etage_appartement                        

In [32]:
# 1. Dimensions
print("Shape :", df.shape)

# 2. Répartition de la cible
print(df["etiquette_dpe"].value_counts(dropna=False))
print(df["etiquette_dpe"].value_counts(normalize=True, dropna=False) * 100)

# 3. Valeurs manquantes
missing_rate = df.isna().mean().sort_values(ascending=False) * 100
missing_rate.head(30)

Shape : (336178, 46)
etiquette_dpe
D    122327
C    114537
E     53684
F     21416
G     10899
B     10442
A      2873
Name: count, dtype: int64
etiquette_dpe
D    36.387568
C    34.070344
E    15.968921
F     6.370435
G     3.242032
B     3.106093
A     0.854607
Name: proportion, dtype: float64


nombre_niveau_immeuble                          78.409355
surface_habitable_immeuble                      70.829442
type_generateur_chauffage_principal_ecs         55.035725
type_generateur_chauffage_principal             54.911386
qualite_isolation_plancher_haut_comble_perdu    53.048683
nombre_appartement                              52.910363
annee_construction                              51.509617
type_installation_ecs                           49.758164
type_installation_chauffage                     49.758164
numero_etage_appartement                        12.850335
qualite_isolation_plancher_bas                   3.794121
configuration_installation_ecs_n1                0.354277
type_energie_generateur_n1_ecs_n1                0.354277
type_generateur_n1_ecs_n1                        0.354277
type_installation_solaire_n1                     0.354277
type_installation_ecs_n1                         0.354277
usage_generateur_n1_ecs_n1                       0.354277
volume_stockag

In [33]:
TARGET = "etiquette_dpe"

HIGH_MISSING_COLUMNS_TO_DROP = [
    "nombre_niveau_immeuble",
    "surface_habitable_immeuble",
    "type_generateur_chauffage_principal_ecs",
    "type_generateur_chauffage_principal",
    "nombre_appartement",
    "annee_construction",
]

TECHNICAL_COLUMNS_TO_DROP = [
    "numero_dpe",
    "date_etablissement_dpe",
    "code_region_ban",
]

columns_to_drop = HIGH_MISSING_COLUMNS_TO_DROP + TECHNICAL_COLUMNS_TO_DROP

df_model = df.drop(columns=columns_to_drop, errors="ignore").copy()

df_model = df_model.dropna(subset=[TARGET])

X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

print("X shape :", X.shape)
print("y shape :", y.shape)
print(y.value_counts(normalize=True) * 100)

X shape : (336178, 36)
y shape : (336178,)
etiquette_dpe
D    36.387568
C    34.070344
E    15.968921
F     6.370435
G     3.242032
B     3.106093
A     0.854607
Name: proportion, dtype: float64


In [34]:
categorical_columns = X.select_dtypes(include=["object", "string"]).columns

X[categorical_columns] = X[categorical_columns].fillna("Non renseigné")

In [35]:
numeric_columns = X.select_dtypes(include=["int64", "float64"]).columns

for col in numeric_columns:
    X[col] = X[col].fillna(X[col].median())

In [36]:
import pandas as pd
import numpy as np

TARGET = "etiquette_dpe"

# Colonnes à ne pas utiliser pour entraîner
TECHNICAL_COLUMNS_TO_DROP = [
    "numero_dpe",
    "date_etablissement_dpe",
    "code_region_ban",  # inutile si tu travailles uniquement sur Hauts-de-France
]

df_model = df.copy()

# Supprimer les lignes sans target
df_model = df_model.dropna(subset=[TARGET])

# Supprimer les colonnes techniques
df_model = df_model.drop(columns=TECHNICAL_COLUMNS_TO_DROP, errors="ignore")

print("Shape après nettoyage :", df_model.shape)
print(df_model[TARGET].value_counts())

Shape après nettoyage : (336178, 43)
etiquette_dpe
D    122327
C    114537
E     53684
F     21416
G     10899
B     10442
A      2873
Name: count, dtype: int64


In [37]:
missing_rate = df_model.isna().mean().sort_values(ascending=False) * 100

columns_to_drop = missing_rate[missing_rate > 75].index.tolist()

# Ne jamais supprimer la target
columns_to_drop = [col for col in columns_to_drop if col != TARGET]

print("Colonnes supprimées car trop vides :")
print(columns_to_drop)

df_model = df_model.drop(columns=columns_to_drop, errors="ignore")

print("Nouvelle shape :", df_model.shape)

Colonnes supprimées car trop vides :
['nombre_niveau_immeuble']
Nouvelle shape : (336178, 42)


In [38]:
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

print("X shape :", X.shape)
print("y shape :", y.shape)

print(y.value_counts(normalize=True) * 100)

X shape : (336178, 41)
y shape : (336178,)
etiquette_dpe
D    36.387568
C    34.070344
E    15.968921
F     6.370435
G     3.242032
B     3.106093
A     0.854607
Name: proportion, dtype: float64


In [39]:
numeric_columns = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_columns = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print("Colonnes numériques :", numeric_columns)
print("Nombre de colonnes numériques :", len(numeric_columns))

print("Colonnes catégorielles :", categorical_columns)
print("Nombre de colonnes catégorielles :", len(categorical_columns))

Colonnes numériques : ['code_departement_ban', 'code_postal_ban', 'code_insee_ban', 'annee_construction', 'surface_habitable_immeuble', 'nombre_appartement', 'numero_etage_appartement', 'hauteur_sous_plafond', 'volume_stockage_generateur_n1_ecs_n1', 'ventilation_posterieure_2012', 'production_electricite_pv_kwhep_par_an']
Nombre de colonnes numériques : 11
Colonnes catégorielles : ['nom_commune_ban', 'zone_climatique', 'classe_altitude', 'type_batiment', 'methode_application_dpe', 'periode_construction', 'classe_inertie_batiment', 'qualite_isolation_enveloppe', 'qualite_isolation_murs', 'qualite_isolation_menuiseries', 'qualite_isolation_plancher_bas', 'qualite_isolation_plancher_haut_comble_perdu', 'type_installation_chauffage', 'type_installation_chauffage_n1', 'configuration_installation_chauffage_n1', 'type_generateur_chauffage_principal', 'type_generateur_n1_installation_n1', 'type_energie_principale_chauffage', 'type_energie_generateur_n1_installation_n1', 'type_emetteur_installa

In [42]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train :", X_train.shape)
print("Test :", X_test.shape)

print(y_train.value_counts(normalize=True) * 100)
print(y_test.value_counts(normalize=True) * 100)

Train : (268942, 41)
Test : (67236, 41)
etiquette_dpe
D    36.387400
C    34.070543
E    15.968871
F     6.370519
G     3.241963
B     3.106246
A     0.854459
Name: proportion, dtype: float64
etiquette_dpe
D    36.388244
C    34.069546
E    15.969124
F     6.370099
G     3.242311
B     3.105479
A     0.855197
Name: proportion, dtype: float64


In [43]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Non renseigné")),
    ("encoder", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=100,
        sparse_output=True
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_columns),
        ("cat", categorical_transformer, categorical_columns)
    ],
    remainder="drop"
)

In [44]:
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix

dummy_model = DummyClassifier(strategy="most_frequent")

dummy_model.fit(X_train, y_train)
y_pred_dummy = dummy_model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred_dummy))
print("Balanced accuracy :", balanced_accuracy_score(y_test, y_pred_dummy))
print("F1 macro :", f1_score(y_test, y_pred_dummy, average="macro"))

print(classification_report(y_test, y_pred_dummy))

Accuracy : 0.36388244392884767
Balanced accuracy : 0.14285714285714285
F1 macro : 0.07622827980072097


D:\UTC\GI04\AI28\PROJET\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

           A       0.00      0.00      0.00       575
           B       0.00      0.00      0.00      2088
           C       0.00      0.00      0.00     22907
           D       0.36      1.00      0.53     24466
           E       0.00      0.00      0.00     10737
           F       0.00      0.00      0.00      4283
           G       0.00      0.00      0.00      2180

    accuracy                           0.36     67236
   macro avg       0.05      0.14      0.08     67236
weighted avg       0.13      0.36      0.19     67236



D:\UTC\GI04\AI28\PROJET\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\UTC\GI04\AI28\PROJET\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [47]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="saga",
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

y_pred_logistic = logistic_model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred_logistic))
print("Balanced accuracy :", balanced_accuracy_score(y_test, y_pred_logistic))
print("F1 macro :", f1_score(y_test, y_pred_logistic, average="macro"))

print(classification_report(y_test, y_pred_logistic))

D:\UTC\GI04\AI28\PROJET\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Accuracy : 0.5439050508656077
Balanced accuracy : 0.6109919121562025
F1 macro : 0.5041495639278615
              precision    recall  f1-score   support

           A       0.53      0.89      0.66       575
           B       0.32      0.72      0.44      2088
           C       0.75      0.65      0.70     22907
           D       0.63      0.48      0.55     24466
           E       0.37      0.42      0.39     10737
           F       0.27      0.41      0.32      4283
           G       0.34      0.71      0.46      2180

    accuracy                           0.54     67236
   macro avg       0.46      0.61      0.50     67236
weighted avg       0.59      0.54      0.56     67236



In [46]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

TARGET = "etiquette_dpe"

# Colonnes à retirer
columns_to_drop = [
    "numero_dpe",
    "date_etablissement_dpe",
    "code_region_ban",
]

df_model = df.drop(columns=columns_to_drop, errors="ignore").copy()
df_model = df_model.dropna(subset=[TARGET])

X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

# Convertir les colonnes object en string pour CatBoost
cat_features = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

for col in cat_features:
    X[col] = X[col].fillna("Non renseigné").astype(str)

# Les numériques : on peut laisser NaN, CatBoost les gère
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Split stratifié sur tout le dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Pondération des classes pour gérer le déséquilibre A/B/F/G
classes = np.unique(y_train)

class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, class_weights_values))

model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    loss_function="MultiClass",
    eval_metric="TotalF1",
    class_weights=class_weights,
    random_seed=42,
    verbose=100,
    allow_writing_files=False
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    early_stopping_rounds=50
)

y_pred = model.predict(X_test).ravel()

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Balanced accuracy :", balanced_accuracy_score(y_test, y_pred))
print("F1 macro :", f1_score(y_test, y_pred, average="macro"))

print(classification_report(y_test, y_pred))

ModuleNotFoundError: No module named 'catboost'